# Chapter 12 — The stellar halo in Gaia data: the Gaia-Sausage/Enceladus merger

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/12_stellar_halo.ipynb)

**Goal.** Find the debris of the last major merger of the Milky Way in the kinematics of metal-poor stars. In 2018
Gaia showed that most of the stellar halo near the Sun moves on very radial orbits: the "Gaia Sausage" (Belokurov et
al. 2018), the remains of a dwarf galaxy, "Gaia-Enceladus" (Helmi et al. 2018), that merged with the Milky Way
8–11 Gyr ago. We reproduce the main observations with Gaia DR3 and AGAMA.

**You will learn**
1. to select halo stars by chemistry, and to test the selection for contamination with kinematics;
2. to separate overlapping populations with a **Gaussian mixture model**, and measure the **velocity anisotropy** $\beta$
   (Chapter 7) of each as a function of metallicity;
3. to recognise a **globular cluster** in phase space, and confirm it against a catalogue;
4. to map stars into **integrals of motion** — energy, angular momentum, actions (Chapters 5 and 9) — and estimate
   orbital eccentricities with AGAMA, checked against orbit integration.

**Data.** The file `halo_gse` of the tutorial's Zenodo record (https://doi.org/10.5281/zenodo.23627795): 285 683 Gaia DR3
stars with line-of-sight velocities, parallax over error above 5, and metallicity $[{\rm M/H}] < -1$ from the Gaia XP
spectra (Andrae, Rix & Chandra 2023). The selection uses chemistry only — no kinematic cut — so that kinematics can
be studied without bias. We write $[{\rm Fe/H}]$ for their $[{\rm M/H}]$; the two agree to about 0.02 dex (see the README).
As we will see in §12.1, these metallicities are reliable only for red giants, and we will select those.

**Units.** kpc, km/s, $M_\odot$; energies in (km/s)$^2$, actions and angular momenta in kpc km/s.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object

import os, hashlib, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Zenodo record 10.5281/zenodo.23627795 (version 1.0); as in Chapter 11
ZENODO = 'https://zenodo.org/records/23627795/files/{}?download=1'
MD5 = {'solar_neighbourhood': 'f6119c98ae50ded14fab91d1d4c071fd', 'halo_gse': '1c0533ce89d6d95ff67406dcf2a95f31',
       'disc_giants': '12f6a9ca0ac3d8ecdcd71988267b8ea7', 'rotation_curves': 'a41c95e4ecc31b3ab3bde99880736c6e',
       'dsph_gaia': '0b135e7f08816209b379f892196307b2'}


def load_data(name, cache=Path('data')):
    """Download data/<name>.parquet from Zenodo once, check its MD5, and return it as a pandas DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} from Zenodo ...')
        urllib.request.urlretrieve(ZENODO.format(path.name), path)
    md5 = hashlib.md5(path.read_bytes()).hexdigest()
    assert md5 == MD5[name], f'{path} is corrupt (md5 {md5}); delete it and rerun'
    return pd.read_parquet(path)


# Solar parameters and the coordinate transformation of Chapter 11 §11.1
R0, z_sun = 8.122, 0.0208                  # kpc
v_sun = (12.9, 245.6, 7.78)                 # km/s
K = 4.740470463533348                       # km/s per (mas/yr * kpc)


def galactocentric(ra, dec, parallax, pmra, pmdec, vlos):
    """Gaia observables (deg, deg, mas, mas/yr, mas/yr, km/s) -> Galactocentric x, y, z [kpc] and vx, vy, vz [km/s]."""
    l, b, pml, pmb = agama.transformCelestialCoords(agama.fromICRStoGalactic, np.radians(ra), np.radians(dec), pmra, pmdec)
    return agama.getGalactocentricFromGalactic(l, b, 1 / parallax, pml * K, pmb * K, vlos,
                                               galcen_distance=R0, z_sun=z_sun, galcen_v_sun=v_sun)

## 12.1 A chemically selected halo sample, and its contaminants

The halo is a spheroid, so its natural coordinates are **spherical**: the Galactocentric distance $r$, the polar angle
$\theta$ (from the North Galactic Pole) and the azimuth $\phi$. The velocity components are

$$ v_r = \frac{\mathbf x \cdot \mathbf v}{r}, \qquad v_\theta = \frac{z v_R - R v_z}{r}, \qquad v_\phi = -\frac{L_z}{R}, $$

where $R$ and $v_R$ are the cylindrical radius and radial velocity, and $v_\phi > 0$ means rotation with the disk
(the sign convention of Chapter 11).

In [ ]:
h = load_data('halo_gse')
cols = ['ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity']
x, y, z, vx, vy, vz = galactocentric(*[h[c].to_numpy(dtype=float) for c in cols])
xv = np.column_stack([x, y, z, vx, vy, vz])        # (N, 6): the input format of AGAMA

r = np.sqrt(x**2 + y**2 + z**2)
R = np.hypot(x, y)
vR = (x * vx + y * vy) / R
vr = (x * vx + y * vy + z * vz) / r
vtheta = (z * vR - R * vz) / r
Lz = x * vy - y * vx                               # negative for disk stars in this frame (Chapter 11)
vphi = -Lz / R
d = 1 / h.parallax.to_numpy(float)
feh = h.feh.to_numpy(float)
print(f'{len(h):,} stars; heliocentric distance: median {np.median(d):.1f} kpc, 90% below {np.percentile(d, 90):.1f} kpc')
print(f'[Fe/H]: 5, 50, 95 percentiles {np.round(np.percentile(feh, [5, 50, 95]), 2)}')

A halo star does not share the disk's rotation. Yet when we look at the most metal-poor stars, $[{\rm Fe/H}] < -1.6$, within
4 kpc of the Sun, many of them move exactly like thin-disk stars: $v_\phi$ near 230 km/s and $|v_z|$ small (Figure 12.1,
left). We count such "cold disk" stars ($|v_\phi - 233| < 40$ and $|v_z| < 40$ km/s) as a function of height above the
plane, and look at where they sit in the Kiel diagram ($T_{\rm eff}$ against $\log g$, both also from the XP spectra).

In [ ]:
cold = (np.abs(vphi - 233) < 40) & (np.abs(vz) < 40)
mp = (feh < -1.6) & (d < 4)
teff, logg = h.teff.to_numpy(float), h.logg.to_numpy(float)
G = h.phot_g_mean_mag.to_numpy(float)
# Galactic latitude, for the cut |b| > 10 deg below (transformCelestialCoords also works without proper motions)
l_gal, b_gal = agama.transformCelestialCoords(agama.fromICRStoGalactic, np.radians(h.ra.to_numpy(float)), np.radians(h.dec.to_numpy(float)))
b_gal = np.degrees(b_gal)
# the vetted red-giant selection of Andrae et al. (2023, their Table 2): bright, cool, low-gravity stars
giant = (G < 16) & (teff < 5200) & (logg < 3.5)
z_edges = [0, 0.2, 0.5, 1.0, 3.0]
frac_z = {name: [cold[mp & sel & (np.abs(z) >= a) & (np.abs(z) < b)].mean() for a, b in zip(z_edges[:-1], z_edges[1:])]
          for name, sel in [('all stars', np.ones(len(h), bool)), ('giants', giant), ('giants, |b| > 10', giant & (np.abs(b_gal) > 10))]}

fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
bins = np.linspace(-300, 400, 141)
ax[0].hist(vphi[mp & (np.abs(z) < 0.2)], bins, histtype='step', lw=2, density=True, label='$|z| < 0.2$ kpc')
ax[0].hist(vphi[mp & (np.abs(z) > 0.5)], bins, histtype='step', lw=2, density=True, label='$|z| > 0.5$ kpc')
ax[0].set_xlabel('$v_\\phi$ [km/s]'); ax[0].set_ylabel('density'); ax[0].legend(); ax[0].set_title('[Fe/H] < -1.6, d < 4 kpc')
xb = np.arange(4)
for k, (name, c) in enumerate(zip(frac_z, ['0.6', 'C0', 'C1'])):
    ax[1].bar(xb + 0.27 * (k - 1), frac_z[name], 0.27, color=c, label=name)
ax[1].set_xticks(xb, [f'{a}-{b}' for a, b in zip(z_edges[:-1], z_edges[1:])]); ax[1].legend()
ax[1].set_xlabel('$|z|$ [kpc]'); ax[1].set_ylabel('fraction of cold disk stars')
ax[2].hist2d(teff[mp], logg[mp], bins=100, range=[[3500, 7500], [0, 5]], norm=LogNorm(), cmap='Greys')
sc = mp & cold & (np.abs(z) < 0.2)
ax[2].plot(teff[sc], logg[sc], ',', color='C3', alpha=0.5)
ax[2].plot([3500, 5200, 5200], [3.5, 3.5, 0], 'C0-', lw=2)                    # the giant box
ax[2].set_xlim(7500, 3500); ax[2].set_ylim(5, 0)
ax[2].set_xlabel('$T_{\\rm eff}$ [K]'); ax[2].set_ylabel('$\\log g$'); ax[2].set_title('red: cold disk stars at $|z| < 0.2$ kpc')
plt.tight_layout(); plt.show()
hot = (teff[sc] > 5500).mean()
for name in frac_z:
    print(f'cold fraction by |z|, {name:16s}:', np.round(frac_z[name], 3))
print(f'cold disk stars at |z| < 0.2 kpc: {sc.sum()}, of which {hot:.0%} have Teff > 5500 K and {giant[sc].mean():.0%} pass the giant cuts')
print(f'the cold giants near the plane: median |b| = {np.median(np.abs(b_gal[sc & giant])):.1f} deg')
assert frac_z['all stars'][0] > 0.3 and hot > 0.7 and frac_z['giants, |b| > 10'][0] < 0.1

**Figure 12.1.** Stars with $[{\rm Fe/H}] < -1.6$ within 4 kpc. Left: distribution of $v_\phi$ near the plane ($|z| < 0.2$ kpc)
and away from it ($|z| > 0.5$ kpc). Middle: fraction of stars with cold, thin-disk kinematics as a function of height $|z|$,
for all stars (grey), giants (blue), and giants at Galactic latitude $|b| > 10°$ (orange). Right: Kiel diagram ($T_{\rm eff}$ against $\log g$) of all these stars
(grey, log-scaled counts), with the cold disk stars near the plane in red; the blue lines bound the giant selection.

Near the plane, 60% of the "very metal-poor" stars have thin-disk kinematics (Figure 12.1, middle, grey). They are not
very metal-poor at all. Most are hotter than 5500 K (right): hot dwarfs and turn-off stars of the disk, whose spectra have
weak metal lines. The XP metallicities of Andrae et al. (2023) were learned from APOGEE stars, almost all cooler than
7000 K, and the model returns a low metallicity for these hot stars that it was never trained on. Zhang, Ardern-Arentsen
& Belokurov (2024, their Appendix A) checked such stars against LAMOST spectra: many are metal-rich stars hotter than
10 000 K. In the Kiel diagram (right) they fall where few real stars live, between the main sequence and the
horizontal branch. Andrae et al. recommend a vetted sample of bright red giants ($G < 16$, $T_{\rm eff} < 5200$ K, $\log g < 3.5$),
for which the metallicities are reliable down to $[{\rm Fe/H}] \approx -3$. With the giants only, the cold fraction near the plane
drops from 60% to 33% (blue bars). The cold giants that remain lie within a few degrees of the Galactic plane, behind
thick dust, which reddens the XP spectra; away from the plane ($|b| > 10°$, orange) the cold fraction is a few per cent
at every height, the level expected from genuine metal-poor stars on disk-like orbits.

The lesson is general: **a sample selected on one quantity must be checked with another**. Here kinematics exposed an
error in the chemistry. Following Zhang et al. (2024), we keep giants at $|b| > 10°$ with parallax over error above 10
(distances better than 10%). We make no cut in $|z|$ or in velocity: the disk stars that
remain are genuine, and we will separate them from the halo with a model rather than with a cut.

In [ ]:
clean = giant & (h.parallax.to_numpy(float) / h.parallax_error.to_numpy(float) > 10) & (np.abs(b_gal) > 10)
print(f'clean sample: {clean.sum():,} giants; heliocentric distance: median {np.median(d[clean]):.1f} kpc, '
      f'90% below {np.percentile(d[clean], 90):.1f} kpc')

## 12.2 The Gaia Sausage

Figure 12.2 shows the clean sample in the $(v_r, v_\phi)$ plane in three metallicity bins.

In [ ]:
feh_bins = [(-1.3, -1.0), (-1.6, -1.3), (-3.0, -1.6)]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for a, (lo, hi) in zip(ax, feh_bins):
    s = clean & (feh > lo) & (feh < hi)
    a.hist2d(vr[s], vphi[s], bins=120, range=[[-400, 400], [-300, 350]], norm=LogNorm(), cmap='Greys')
    a.set_title(f'{lo} < [Fe/H] < {hi}: {s.sum():,} stars')
    a.set_xlabel('$v_r$ [km/s]')
ax[0].set_ylabel('$v_\\phi$ [km/s]')
plt.tight_layout(); plt.show()

**Figure 12.2.** Spherical radial velocity $v_r$ against rotation velocity $v_\phi$ for the clean sample of giants in three
metallicity bins; log-scaled counts.

Several populations overlap in Figure 12.2. A **rotating** one, centred near $v_\phi \approx 150$ km/s with small $v_r$: the
metal-weak tail of the thick disk and the "Splash" (Belokurov et al. 2020), most prominent in the most metal-rich bin.
A **non-rotating** one, with $v_\phi \approx 0$ but $v_r$ spreading to $\pm 300$ km/s: the Sausage. Its two lobes at
$v_r \approx \pm 200$ km/s are stars of the same debris near the Sun moving out of and into the inner Galaxy on nearly
radial orbits; few are at rest in $v_r$, because the Sun sits near the middle of their long radial excursions, where
they move fastest (compare the radial velocity of the orbits in Chapter 3). And, most visible at the lowest
metallicities (right), a **round**, roughly isotropic cloud. The Sausage is clearest at $-1.6 < [{\rm Fe/H}] < -1.3$ (middle).

The **anisotropy parameter** (Chapter 7)

$$ \beta = 1 - \frac{\sigma_\theta^2 + \sigma_\phi^2}{2\sigma_r^2} $$

quantifies the shape of a velocity distribution: $\beta = 0$ for isotropic orbits, $\beta \to 1$ for purely radial ones. The
simplest measurement is one $\beta$ per metallicity bin from the plain variances of all stars in it. But a bin that
mixes a rotating disk, a radial Sausage and a round halo gives a number that describes none of them: the disk's
rotation, spread over the bin, inflates $\sigma_\phi$, and the round halo dilutes the radial one.

### A Gaussian mixture model

Instead we describe the velocities in each bin as a sum of $K$ Gaussians, one per population,

$$ p(\mathbf v) = \sum_{k=1}^K f_k \, \mathcal N(\mathbf v \mid \boldsymbol\mu_k, \boldsymbol\Sigma_k), \qquad \sum_k f_k = 1, $$

where $\mathbf v = (v_r, v_\theta, v_\phi)$, $f_k$ is the fraction of stars in component $k$, and $\boldsymbol\Sigma_k$ is diagonal with
entries $(\sigma_{r,k}^2, \sigma_{\theta,k}^2, \sigma_{\phi,k}^2)$. Every component then has its own $\beta_k$. We use $K = 3$ (a disk, a radial
and a round component) and two physical constraints: in a steady state no population streams in $r$ or $\theta$, so
$\mu_r = \mu_\theta = 0$ for every component; only the mean rotation $\mu_\phi$ is free. Without these constraints the fit splits
the Sausage into its two lobes at $\pm 200$ km/s, which is a better description of the velocities but does not give
one $\beta$ for the debris (Exercise 1).

The parameters are found with the **expectation–maximisation (EM)** algorithm, which repeats two steps until the
likelihood stops growing:

- *E-step*: for each star $i$, the probability that it belongs to component $k$,
  $p_{ik} = f_k \mathcal N(\mathbf v_i \mid k) / \sum_j f_j \mathcal N(\mathbf v_i \mid j)$;
- *M-step*: each component's parameters become the $p_{ik}$-weighted fraction, mean and variance of the stars:
  $f_k = \langle p_{ik} \rangle_i$, $\mu_{\phi,k} = \sum_i p_{ik} v_{\phi,i} / \sum_i p_{ik}$, $\sigma^2_{r,k} = \sum_i p_{ik} v_{r,i}^2 / \sum_i p_{ik}$, and so on.

Each iteration cannot decrease the likelihood, but EM can stop in a local maximum, so we start it from three different
guesses and keep the best. Velocity errors (6–10 km/s for these stars, against dispersions of 70–190 km/s) are
neglected; a model that includes them is "extreme deconvolution" (Bovy, Hogg & Roweis 2011).

In [ ]:
V = np.array([vr, vtheta, vphi])          # (3, N): the three velocity components of every star


def mixture(X, f, mu_phi, sig, n_iter=500, tol=1e-6):
    """EM fit of a sum of K Gaussians to velocities X of shape (3, N), with mean v_r = v_theta = 0 for every component.

    f: (K,) fractions; mu_phi: (K,) mean v_phi; sig: (K, 3) dispersions (sigma_r, sigma_theta, sigma_phi) -- the starting
    guesses. Returns the fitted (f, mu_phi, sig) and the log-likelihood per star."""
    f, mu_phi, sig = np.array(f, float), np.array(mu_phi, float), np.array(sig, float)
    old = -np.inf
    for _ in range(n_iter):
        mu = np.zeros((len(f), 3, 1)); mu[:, 2, 0] = mu_phi                # (K, 3, 1): means, broadcast over stars
        # E-step: log of f_k N(v_i | k) for every component and star, shape (K, N); constants of N dropped
        lnp = np.log(f)[:, None] - np.log(sig).sum(1)[:, None] - 0.5 * (((X - mu) / sig[:, :, None])**2).sum(1)
        lnL = np.logaddexp.reduce(lnp, axis=0)                           # log sum_k, computed without overflow
        p = np.exp(lnp - lnL)                                            # membership probabilities, each column sums to 1
        # M-step: weighted fractions, mean rotations and dispersions
        w = p.sum(1)
        f = w / X.shape[1]
        mu_phi = (p * X[2]).sum(1) / w
        mu[:, 2, 0] = mu_phi
        sig = np.sqrt((p[:, None, :] * (X - mu)**2).sum(2) / w[:, None])
        if lnL.mean() - old < tol:
            break
        old = lnL.mean()
    return f, mu_phi, sig, lnL.mean()


def beta_of(sig):
    """Anisotropy from dispersions (..., 3) ordered (sigma_r, sigma_theta, sigma_phi)."""
    return 1 - (sig[..., 1]**2 + sig[..., 2]**2) / (2 * sig[..., 0]**2)


starts = [([0.4, 0.3, 0.3], [0, 0, 150], [[180, 60, 60], [120, 120, 120], [60, 50, 40]]),   # (f, mu_phi, sig) guesses
          ([0.6, 0.2, 0.2], [0, 0, 170], [[170, 70, 70], [140, 140, 140], [50, 40, 30]]),
          ([0.2, 0.6, 0.2], [0, 0, 120], [[190, 50, 50], [130, 120, 120], [80, 60, 50]])]


def fit_bin(X):
    """Best of the three starts; components reordered as (radial, round, rotating)."""
    f, mu_phi, sig, lnL = max((mixture(X, *s) for s in starts), key=lambda r: r[3])
    rot = np.argmax(mu_phi)                                   # the rotating component: largest mean v_phi
    rest = [k for k in range(3) if k != rot]
    rad, rnd = sorted(rest, key=lambda k: -beta_of(sig[k]))   # of the other two: the radial one has the larger beta
    order = [rad, rnd, rot]
    return f[order], mu_phi[order], sig[order]


edges = np.array([-3.5, -2.4, -2.0, -1.8, -1.6, -1.4, -1.2, -1.0])
names = ['radial (Sausage)', 'round halo', 'rotating']
centres, fits, B_naive = [], [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    s = clean & (feh > lo) & (feh < hi)
    centres.append(np.median(feh[s]))
    var = V[:, s].var(axis=1)                                          # the naive, single-population measurement
    B_naive.append(1 - (var[1] + var[2]) / (2 * var[0]))
    fits.append(fit_bin(V[:, s]))
    f, mu_phi, sig = fits[-1]
    print(f'{lo:5.1f} < [Fe/H] < {hi:5.1f}: N = {s.sum():6,}, naive beta = {B_naive[-1]:.2f} | ' +
          ' | '.join(f'{n}: f = {f[k]:.2f}, <v_phi> = {mu_phi[k]:4.0f}, beta = {beta_of(sig[k]):.2f}' for k, n in enumerate(names)))
F = np.array([q[0] for q in fits]); MU = np.array([q[1] for q in fits]); SIG = np.array([q[2] for q in fits])
B = beta_of(SIG)                                                       # (n_bins, 3)

The fit takes a few seconds per bin. To estimate the uncertainties we refit bootstrap resamples of the stars, starting
EM from the best fit (which converges in a few iterations).

In [ ]:
rng = np.random.default_rng(0)
B_err = np.zeros_like(B); F_err = np.zeros_like(F)
for j, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
    idx = np.flatnonzero(clean & (feh > lo) & (feh < hi))
    boot = [mixture(V[:, rng.choice(idx, len(idx))], F[j], MU[j], SIG[j], n_iter=100)[:3] for _ in range(20)]
    B_err[j] = np.std([beta_of(b[2]) for b in boot], axis=0)
    F_err[j] = np.std([b[0] for b in boot], axis=0)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
colors = ['C3', 'C0', 'C2']
for k in range(3):
    ax[0].errorbar(centres, B[:, k], B_err[:, k], fmt='o-', color=colors[k], label=names[k])
    ax[1].errorbar(centres, F[:, k], F_err[:, k], fmt='o-', color=colors[k])
    ax[2].plot(centres, MU[:, k], 'o-', color=colors[k])
ax[0].plot(centres, B_naive, 'k--', label='all stars, one $\\beta$')
ax[0].set_ylabel('$\\beta$'); ax[0].set_ylim(-0.2, 1); ax[0].legend(fontsize=8)
ax[1].set_ylabel('fraction of stars')
ax[2].set_ylabel('mean $v_\\phi$ [km/s]')
for a in ax:
    a.set_xlabel('[Fe/H]')
plt.tight_layout(); plt.show()

**Figure 12.3.** Three-component Gaussian mixture fits of the velocities of the clean sample in metallicity bins. Left:
anisotropy $\beta$ of each component (bootstrap errors), and the single $\beta$ of all stars in the bin (dashed). Middle: the
fraction of stars in each component. Right: mean rotation velocity of each component.

Is a sum of three Gaussians a fair description? Figure 12.4 compares the distributions of the three velocity components
in one bin with the model.

In [ ]:
j = 4                                                                  # the bin -1.6 < [Fe/H] < -1.4
s = clean & (feh > edges[j]) & (feh < edges[j + 1])
vgrid = np.linspace(-450, 450, 451)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for c, (a, label) in enumerate(zip(ax, ['$v_r$', '$v_\\theta$', '$v_\\phi$'])):
    a.hist(V[c, s], np.linspace(-450, 450, 91), density=True, histtype='step', color='k', lw=1.5, label='data')
    total = 0
    for k in range(3):
        mean = MU[j, k] if c == 2 else 0.0
        g = F[j, k] * np.exp(-0.5 * ((vgrid - mean) / SIG[j, k, c])**2) / (np.sqrt(2 * np.pi) * SIG[j, k, c])
        a.plot(vgrid, g, color=colors[k], lw=1, label=names[k]); total = total + g
    a.plot(vgrid, total, 'k:', lw=2, label='model')
    a.set_xlabel(label + ' [km/s]')
ax[0].set_ylabel('density'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

# the measurements this section rests on
rich = np.array(centres) > -2.0
assert np.all(B[rich, 0] > 0.75) and np.all(B[:, 1] < 0.6)        # radial Sausage vs a round halo
assert F[0, 0] < F[4, 0] and F[0, 1] > F[4, 1]                     # the Sausage dominates at [Fe/H] ~ -1.5, not at < -2.4
assert max(B_naive) < 0.75                                          # one beta per bin underestimates the anisotropy

**Figure 12.4.** Distributions of $v_r$, $v_\theta$ and $v_\phi$ for $-1.6 < [{\rm Fe/H}] < -1.4$ (black histograms), the three components
of the mixture model (coloured) and their sum (dotted).

The mixture model separates what one $\beta$ per bin mixed up (Figure 12.3). For $[{\rm Fe/H}] > -2$ the largest component is
strongly radial, $\beta \approx 0.80$–0.86, with no rotation: the Sausage. Next to it are a round halo
($\beta \approx 0.2$–0.55, slightly retrograde) and a rotating component, the disk. The radial component makes up about 60%
of the stars at $-1.6 < [{\rm Fe/H}] < -1.2$; above that the disk takes over. Below $[{\rm Fe/H}] \approx -2$ the round halo grows to
35–40% of the stars, and the most radial component becomes less radial ($\beta = 0.67$, then 0.47) and starts to rotate:
the Sausage fades into the older, more isotropic halo. At these metallicities the rotating component, with
$\langle v_\phi \rangle \approx 130$ km/s and large dispersions, is not a disk but the "prograde halo" that Zhang et al. (2024) found among
very metal-poor stars. The single $\beta$ of all stars (dashed) peaks at only 0.66: it describes no real population.

Belokurov et al. (2018), with a similar mixture model of SDSS stars, found the same sharp transition: $\beta \approx 0.2$–0.4 for
the halo below $[{\rm Fe/H}] \approx -1.7$ and $\beta \approx 0.8$–0.9 above it. Their interpretation: the radially anisotropic stars are the
debris of one massive dwarf galaxy, whose orbit was radialised before it was disrupted — the last major merger of the
Milky Way.

The model describes the data well, but not perfectly (Figure 12.4): the observed $v_r$ distribution has a flatter top and
heavier shoulders at $|v_r| \approx 200$–300 km/s than the sum of Gaussians. These are the two lobes of Figure 12.2, which a single
Gaussian centred on zero cannot follow, and the reason why our $\beta \approx 0.86$ stays below the 0.9 of the literature. Zhang
et al. (2024) describe the Sausage with two Gaussians, one for each lobe (Exercise 1).

## 12.3 A globular cluster in phase space

In the most metal-poor panel of Figure 12.2 there is a compact dark spot near $(v_r, v_\phi) \approx (50, 100)$ km/s. A
single dense knot in velocity space suggests a group of stars moving together: a star cluster. Where are these stars on
the sky?

In [ ]:
knot = clean & (feh < -1.6) & (np.abs(vr - 50) < 15) & (np.abs(vphi - 100) < 15)
ra, dec = h.ra.to_numpy(float), h.dec.to_numpy(float)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# the densest spot on the sky: the knot star with the most knot neighbours within 1 degree
sep = lambda a, b: np.hypot((ra[a] - ra[b]) * np.cos(np.radians(dec[b])), dec[a] - dec[b])   # small-angle separation [deg]
ik = np.flatnonzero(knot)
centre = ik[np.argmax([(sep(ik, i) < 1).sum() for i in ik])]
core = knot & (sep(np.arange(len(h)), centre) < 1)           # knot stars within 1 degree of that spot
ax[0].plot(ra[knot & ~core], dec[knot & ~core], '.', color='0.6', ms=3)
ax[0].plot(ra[core], dec[core], 'r.', ms=3)
ax[0].plot(ra[centre], dec[centre], 'o', ms=16, mfc='none', mec='r', mew=1.5)
ax[0].set_xlabel('RA [deg]'); ax[0].set_ylabel('Dec [deg]'); ax[0].set_xlim(360, 0); ax[0].set_title(f'{knot.sum()} stars of the knot')
ax[1].plot(h.pmra[knot & ~core], h.pmdec[knot & ~core], '.', color='0.6', ms=3, label='knot, elsewhere on the sky')
ax[1].plot(h.pmra[core], h.pmdec[core], 'r.', ms=3, label='knot, within 1° of the densest spot')
ax[1].set_xlabel('$\\mu_{\\alpha*}$ [mas/yr]'); ax[1].set_ylabel('$\\mu_\\delta$ [mas/yr]'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# NGC 6397 in Vasiliev & Baumgardt (2021, table A1, VizieR J/MNRAS/505/5978): position, proper motion, parallax
ngc6397 = dict(ra=265.175, dec=-53.674, pmra=3.260, pmdec=-17.664, parallax=0.416)
for k, ref in ngc6397.items():
    print(f'{k:8s}: median of the core stars {h[k][core].median():8.3f}   catalogue {ref:8.3f}')
print(f'{core.sum()} of the {knot.sum()} knot stars lie within 1 degree of its centre')
assert abs(h.pmra[core].median() - 3.260) < 0.3 and abs(h.pmdec[core].median() + 17.664) < 0.3
assert abs(h.ra[core].median() - 265.175) < 0.3 and abs(h.dec[core].median() + 53.674) < 0.3

**Figure 12.5.** Stars in the velocity-space knot ($[{\rm Fe/H}] < -1.6$, $v_r$ and $v_\phi$ within 15 km/s of (50, 100) km/s).
Left: positions on the sky. Right: proper motions. Red: stars within 1° of the densest spot on the sky (circled).

About half of the knot stars sit in one spot of the sky (Figure 12.5, left, red), and these share one proper motion
(right); the rest are field halo stars that happen to have similar velocities. The proper motion and parallax match the nearby globular cluster **NGC 6397**, about 2.4 kpc away (parallax 0.42 mas). Our median parallax is 0.03 mas smaller: Gaia DR3 parallaxes have a known
zero-point offset of a few hundredths of a mas, which the catalogue corrects (Lindegren et al. 2021) and our file does not. A cluster is bound, so its stars stay together in all six phase-space coordinates; dissolved
debris like the Sausage stays together only in the integrals of motion, which is the subject of the next section. The
cluster is a small part of the sample and does not affect the conclusions; we leave it in.

## 12.4 Integrals of motion

Debris from a disrupted satellite spreads along its orbit and phase-mixes (Chapter 6) until it fills a large volume of
space. But its stars keep their **integrals of motion**, approximately those of the satellite: energy $E$, angular momentum
$L_z$ and the actions. We compute them in McMillan's (2017) potential, as in Chapter 11. AGAMA gives the potential at
many points at once with `pot.potential(points)`, and `pot.Rperiapo(xv)` estimates the pericentre and apocentre of each
orbit, from which the eccentricity $e = (r_{\rm apo} - r_{\rm peri}) / (r_{\rm apo} + r_{\rm peri})$.

`Rperiapo` with an $(N, 6)$ array treats the potential as if it were spherical (it uses the total angular momentum), so
for orbits that feel the flattened disk it is an approximation. We check it against the minimum and maximum radius of
300 orbits integrated for 3 time units (2.9 Gyr).

In [ ]:
pot = agama.Potential(os.path.join(os.path.dirname(agama.__file__), 'data', 'McMillan17.ini'))
E = pot.potential(xv[:, :3]) + 0.5 * (vx**2 + vy**2 + vz**2)        # energy per unit mass [(km/s)^2]
peri_apo = pot.Rperiapo(xv)                                        # (N, 2) array
ecc = (peri_apo[:, 1] - peri_apo[:, 0]) / (peri_apo[:, 1] + peri_apo[:, 0])

test = np.random.default_rng(1).choice(np.flatnonzero(clean), 300, replace=False)
orbits = agama.orbit(potential=pot, ic=xv[test], time=3.0, trajsize=3000, verbose=False)
r_orb = [np.linalg.norm(traj[:, :3], axis=1) for times, traj in orbits]   # radius along each orbit
ecc_orb = np.array([(q.max() - q.min()) / (q.max() + q.min()) for q in r_orb])
diff = np.abs(ecc_orb - ecc[test])
print(f'eccentricity: Rperiapo vs orbit integration: median |difference| {np.median(diff):.3f}, 90th percentile {np.percentile(diff, 90):.3f}')
assert np.median(diff) < 0.05

The estimate agrees with the integrated orbits to 0.04 for a typical star, and to about 0.15 for 90% of them; good
enough to separate radial from circular orbits. (Integrating all orbits would be more accurate, and takes a few minutes.)

Now the energy–angular momentum plane (Figure 12.6) and the "action diamond" (Figure 12.7). For the diamond, AGAMA's
`ActionFinder` gives $(J_r, J_z, J_\phi)$; with $J_{\rm tot} = J_r + J_z + |J_\phi|$, the horizontal axis $J_\phi/J_{\rm tot}$ runs
from retrograde ($-1$) to prograde circular orbits (+1), and the vertical axis $(J_z - J_r)/J_{\rm tot}$ from radial orbits
($-1$, bottom corner) to polar ones (+1, top). We flip the sign of $L_z$ and $J_\phi$ so that prograde (disk-like) is positive.

In [ ]:
J = agama.ActionFinder(pot)(xv)
Jr, Jz, Jphi = J.T
Jtot = Jr + Jz + np.abs(Jphi)
radial = ecc > 0.8                                    # our working definition of "Sausage-like" orbits
print(f'stars with NaN actions (unbound or failed): {np.isnan(J).any(axis=1).sum()}')
print(f'clean sample: {np.mean(radial[clean]):.2f} have e > 0.8')

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
s = clean
ax[0].hist2d(-Lz[s], E[s] / 1e5, bins=200, range=[[-3000, 4000], [-2.1, -0.6]], norm=LogNorm(), cmap='Greys')
ax[0].plot(-Lz[s & radial], E[s & radial] / 1e5, ',', color='C3', alpha=0.3)
ax[0].set_xlabel('$-L_z$ [kpc km/s]'); ax[0].set_ylabel('$E$ [$10^5$ (km/s)$^2$]'); ax[0].set_title('red: $e > 0.8$')
H = ax[1].hist2d(-Jphi[s] / Jtot[s], (Jz[s] - Jr[s]) / Jtot[s], bins=120, range=[[-1, 1], [-1, 1]], norm=LogNorm(), cmap='Greys')
ax[1].plot(-Jphi[s & radial] / Jtot[s & radial], (Jz[s & radial] - Jr[s & radial]) / Jtot[s & radial], ',', color='C3', alpha=0.3)
ax[1].set_xlabel('$-J_\\phi / J_{\\rm tot}$'); ax[1].set_ylabel('$(J_z - J_r) / J_{\\rm tot}$')
plt.tight_layout(); plt.show()

**Figure 12.6.** Left: energy against angular momentum (sign flipped so that disk rotation is positive) for the clean
sample; log-scaled counts, with stars on orbits of eccentricity $e > 0.8$ overplotted in red. Right: the action diamond,
$J_\phi/J_{\rm tot}$ against $(J_z - J_r)/J_{\rm tot}$, with the same $e > 0.8$ stars in red.

In Figure 12.6 (left), the disk stars form the dense sequence running to the right along the lower envelope (the
circular orbits have the lowest energy at each $L_z$). The radial orbits ($e > 0.8$, red) form a narrow vertical
column at $L_z \approx 0$ over a wide range of energies: the Sausage. In the action diamond (right), the disk crowds into the right-hand corner (prograde, nearly circular),
and the Sausage into the bottom corner: orbits dominated by $J_r$.

## 12.5 The metallicity of the radial debris

If the radial orbits are the debris of one galaxy, their stars should share that galaxy's chemistry. Figure 12.7 shows,
for each metallicity bin, the fraction of stars on orbits with $e > 0.8$, and the metallicity distributions of stars on
radial and on low-eccentricity orbits.

In [ ]:
frac_radial = [radial[clean & (feh > lo) & (feh < hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(centres, frac_radial, 'ko-')
ax[0].set_xlabel('[Fe/H]'); ax[0].set_ylabel('fraction with $e > 0.8$')
fb = np.linspace(-3, -1, 41)
ax[1].hist(feh[clean & radial], fb, histtype='step', lw=2, density=True, label='$e > 0.8$')
ax[1].hist(feh[clean & (ecc < 0.5)], fb, histtype='step', lw=2, density=True, label='$e < 0.5$')
ax[1].set_xlabel('[Fe/H]'); ax[1].set_ylabel('density'); ax[1].legend()
plt.tight_layout(); plt.show()
print('fraction with e > 0.8 per bin:', np.round(frac_radial, 2))
assert np.argmax(frac_radial) in (3, 4, 5) and max(frac_radial) - frac_radial[0] > 0.05

**Figure 12.7.** Left: fraction of the clean sample on orbits with $e > 0.8$, in the metallicity bins of Figure 12.3.
Right: normalised metallicity distributions of stars with $e > 0.8$ and with $e < 0.5$.

The radial orbits are most common at $-1.6 \lesssim [{\rm Fe/H}] \lesssim -1.2$ (Figure 12.7, left) and rarer among the most metal-poor
stars. Our sample stops at $[{\rm Fe/H}] = -1$, so we see only the metal-poor side of the debris; spectroscopic surveys
that reach higher metallicities find Sausage stars above $-1$ as well. A galaxy that enriched its gas to these levels was
massive for a dwarf: Belokurov et al. (2018) estimate a virial mass above $10^{10}\,M_\odot$ for the progenitor. The
low-eccentricity stars (right) are the
disk-like population of Figure 12.2 and are more metal-rich.

## Summary

- Chemical selections need kinematic checks: 60% of the "very metal-poor" stars near the plane have thin-disk kinematics.
  They are hot disk stars with wrong XP metallicities; restricting the sample to red giants removes them.
- One $\beta$ per metallicity bin mixes populations. A Gaussian mixture model separates a radial component
  ($\beta \approx 0.8$–0.86) — the Gaia Sausage, the debris of the Gaia-Sausage/Enceladus merger — from a round halo
  ($\beta \approx 0.2$–0.5) and the rotating disk. The Sausage dominates at $-1.6 \lesssim [{\rm Fe/H}] \lesssim -1.2$ and fades below $-2$.
- A bound cluster (NGC 6397) is compact in all of phase space; dissolved debris is compact only in the integrals of motion.
- AGAMA gives energies, actions and approximate peri/apocentres for $10^5$ stars in under a second; check approximations
  against orbit integration.

| task | AGAMA |
|---|---|
| potential at many points | `pot.potential(points)` with `points` of shape $(N, 3)$ |
| peri- and apocentre | `pot.Rperiapo(xv)` (spherical approximation for $(N, 6)$ input) or `pot.Rperiapo(E, L)` |
| actions | `agama.ActionFinder(pot)(xv)` → $(J_r, J_z, J_\phi)$ |
| orbit check | `agama.orbit(potential=pot, ic=xv, time=T, trajsize=n, verbose=False)` |

## Exercises

1. **The two lobes.** Extend `mixture` so that the radial component becomes two Gaussians with means $\pm\mu_r$ (free)
   and shared dispersions. Does the fit improve (compare the log-likelihoods), and what is $\beta$ of the pair, computed
   from the total variance $\sigma_r^2 + \mu_r^2$? Then try `sklearn.mixture.GaussianMixture` with free means and
   4–6 components, as Zhang et al. (2024) did, and identify the components.
2. **The potential.** Recompute $E$ and the eccentricities in the halo-fit potential of Chapter 11 §11.4. Which stars
   change their classification $e > 0.8$?
3. **Retrograde debris.** Select stars with $-L_z < -1000$ kpc km/s and high energy. How does their metallicity distribution
   compare with the Sausage's? (This region holds the debris called Sequoia.)
4. **Remove the cluster.** Remove NGC 6397's stars (all stars within 0.5° of its centre) and refit the most metal-poor
   bins. Do the components change?
5. **Hot stars.** Fit the mixture model to the full file, without the giant cuts. Which component absorbs the
   contaminants, and how much does its $\beta$ change?

## References

- R. Andrae, H.-W. Rix & V. Chandra (2023), "Robust data-driven metallicities for 175 million stars from Gaia XP spectra",
  ApJS 267, 8: https://ui.adsabs.harvard.edu/abs/2023ApJS..267....8A
- V. Belokurov et al. (2018), "Co-formation of the disc and the stellar halo", MNRAS 478, 611:
  https://ui.adsabs.harvard.edu/abs/2018MNRAS.478..611B
- V. Belokurov et al. (2020), "The biggest splash", MNRAS 494, 3880: https://ui.adsabs.harvard.edu/abs/2020MNRAS.494.3880B
- A. Helmi et al. (2018), "The merger that led to the formation of the Milky Way's inner stellar halo and thick disk",
  Nature 563, 85: https://ui.adsabs.harvard.edu/abs/2018Natur.563...85H
- L. Lindegren et al. (2021), "Gaia Early Data Release 3. Parallax bias versus magnitude, colour, and position", A&A 649, A4:
  https://ui.adsabs.harvard.edu/abs/2021A&A...649A...4L
- J. Bovy, D. Hogg & S. Roweis (2011), "Extreme deconvolution", Annals of Applied Statistics 5, 1657:
  https://doi.org/10.1214/10-AOAS439
- P. McMillan (2017), MNRAS 465, 76: https://ui.adsabs.harvard.edu/abs/2017MNRAS.465...76M
- E. Vasiliev & H. Baumgardt (2021), "Gaia EDR3 view on galactic globular clusters", MNRAS 505, 5978:
  https://ui.adsabs.harvard.edu/abs/2021MNRAS.505.5978V
- H. Zhang, A. Ardern-Arentsen & V. Belokurov (2024), "On the existence of a very metal-poor disc in the Milky Way",
  MNRAS 533, 889: https://ui.adsabs.harvard.edu/abs/2024MNRAS.533..889Z
- Data: V. Belokurov (2026), "Galactic Dynamics Tutorial data", Zenodo, https://doi.org/10.5281/zenodo.23627795 — please also cite
  the sources listed in its README (Gaia DR3, Andrae et al. 2023, Katz et al. 2023).